# Correlation between Genomic Feature Position and DNA Sequence Information
Does the information contained at a position in a DNA sequence correlate to the position of known genomic features?


## Definitions
This is extremely ambigouous, we need some definitions.

### Binary Derivative
For a binary string *s* of length n,

$$S = s_0, s_1, ... s_{n-1}$$

The binary derivative of S, *dS*, is another binary string given by the XOR value of all adjacent positions in the string *S*.

This means
$$|dS| + 1 = |S|$$

And *dS* is defined as
$$
dS_i = \{ XOR(S_i, S_{i+1}) : i \in [0, n-1) \} 
$$

### DNA Sequence
A finite string of {A, C, G, T} characters.

### GC Content
The proportion of G|C characters in the DNA sequence string. Biologically, it reflects nucletide binding between the DNA double-helix strands https://en.wikipedia.org/wiki/GC-content.


### Binary Entropy
TODO, defined for a binary sequence.

### (DNA) Information
Loosly based on complexity. What is the length _L_ (bits) of the minimum program required to produce the DNA sequence (in binary representation)? The larger _L_ the larger the information.

Note, given a DNA sequence _S_ $$L <= |S|$$

Also if _L == |S|_ then _S_ would be truly, completely random, by definition, and therefore also impossible. [citation needed]

#### DNA Entropy
Binary entropy of a particular encoding of a string of DNA sequence. Entropy is directly proportional to information in this context.

#### DNA Compressability
Compressability of a DNA string, or it's encoded binary representation. When encoded this will depend on the encoding bit size. Compressability is inverserly proportional to information in this context.

# Hypothesis
The information in some region of DNA sequence correlates with some genomic feature.

# Method
Sample some sequences from human. For windows, generate information signals. Compare signals with labels.

## 00 Test data
Get a few samples to help develop the encoding functions.

In [26]:
import gabi.utils as gu
import numpy as np
import pickle
import sys

In [14]:
sgen = gu.sample_generator(n=10, sample_len=10000, feature_types=['exon', 'cds'], biotype='protein_coding' )
test_samples = [s async for s in sgen]
len(test_samples)
[np.sum(s['labels'], axis=0) for s in test_samples]


[array([0, 0]),
 array([0, 0]),
 array([0, 0]),
 array([0, 0]),
 array([0, 0]),
 array([0, 0]),
 array([0, 0]),
 array([0, 0]),
 array([0, 0]),
 array([0, 0])]

In [9]:
#pickle.dump( test_samples, open( "test.p", "wb" ) )
# don't save again, use the saved samples from earlier

In [15]:
saved_samples = pickle.load(open("test.p", "rb"))
[np.sum(s['labels'], axis=0) for s in saved_samples]

## 01 Encode data
Create the encoding functions


In [102]:
# generic encoder wrapper for mapping strings to encoding dicts
def encode(seq, enc):
    """Encode (into numpy array) a sequence str with and enc(oding) dict"""
    return np.array(
        [c for s in seq for c in enc[s]], # flat list
        dtype="uint8"
    )


### GC
_1-bit_

$$
A \rightarrow 0 \\
C \rightarrow 1 \\
G \rightarrow 1 \\ 
T \rightarrow 0
$$


In [103]:
gc = {
   'A': '0',
   'C': '1',
   'G': '1', 
   'T': '0'
} 

encode('ACGT', gc)

array([0, 1, 1, 0], dtype=uint8)

### GC Maximum Entropy
_2-bit_

Relate by the binary derivative. CG, AT related by same binary entropy {0, 1}, the more complex encodings (G,C) which have the higher entropy encodings, are biologically more important or more abundant at least, in coding regions.

Note, this biases GC content with a higher binary entropy, which is based on the 1st to *n-1*th order binary dervitives of a string.

$$
A \rightarrow 00 \\
C \rightarrow 01 \\
G \rightarrow 10 \\ 
T \rightarrow 11
$$

In [104]:
gcme = {
   'A': '00',
   'C': '01',
   'G': '10', 
   'T': '11'
} 

encode('ACGT', gcme)

array([0, 0, 0, 1, 1, 0, 1, 1], dtype=uint8)

### GC Maximum Magnitude
_3-bit_

The binary derivatives are all 1, GC bias to higer magnitude(2).

001
110
100
011

$$
A \rightarrow 001 \\
C \rightarrow 011 \\
G \rightarrow 110 \\ 
T \rightarrow 100
$$

In [105]:
gcmm = {
   'A': '001',
   'C': '011',
   'G': '110', 
   'T': '100'
} 

encode('ACGT', gcmm)

array([0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 0, 0], dtype=uint8)

### Categorical
_4-bit_

A balanced encoding.

_AKA One-Hot encoding_

$$
A \rightarrow 0001 \\
C \rightarrow 0010 \\
G \rightarrow 0100 \\ 
T \rightarrow 1000
$$

In [106]:
categorical = {
   'A': '0001',
   'C': '0010',
   'G': '0100', 
   'T': '1000'
} 

encode('ACGT', categorical)

array([0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0], dtype=uint8)